In [2]:
#imports

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
import pandas as pd
from sklearn.model_selection import train_test_split



In [3]:
# Load the dataset (assuming 'data.csv' is the file name and it contains 'text' and 'label' columns)
df = pd.read_csv('/content/case_type_training_data_2000.csv')
display(df.head())

,id,query,case_type,difficulty
0,1,Find precedents involving case seeking enforce...,Constitutional,Medium
1,2,challenge to a family court order on maintenance,Family,Short
2,3,complaint regarding non-delivery of goods purc...,Consumer,Short
3,4,The query concerns petition challenging restri...,Constitutional,Medium
4,5,The query concerns government employee challen...,Service Law,Medium


In [19]:
# Extract queries and labels
queries = df['query'].tolist()
labels = df['case_type'].tolist()

# Split the data into training and testing sets
train_queries, test_queries, train_labels, test_labels = train_test_split(
    queries, labels, test_size=0.5, random_state=42
)

print(f"Training samples: {len(train_queries)}")
print(f"Testing samples: {len(test_queries)}")

Training samples: 1000
Testing samples: 1000


In [20]:
from sklearn.metrics import classification_report, accuracy_score

#Training
# Load pretrained SBERT
sbert = SentenceTransformer("all-MiniLM-L6-v2")

# Convert training queries to embeddings
X_train = sbert.encode(train_queries)

# Train classifier
classifier = LogisticRegression(
    max_iter=1000
)

classifier.fit(X_train, train_labels);

# Convert test queries to embeddings
X_test = sbert.encode(test_queries)

# Predict on the test set
test_labels_predicted = classifier.predict(X_test)

# Calculate and print accuracy
accuracy = accuracy_score(test_labels, test_labels_predicted)
print(f"Accuracy: {accuracy:.4f}")

# Generate and print a classification report
print("\nClassification Report:")
print(classification_report(test_labels, test_labels_predicted))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Accuracy: 0.9890

Classification Report:
                precision    recall  f1-score   support

   Arbitration       1.00      1.00      1.00       103
         Civil       0.99      0.96      0.97        99
Constitutional       0.96      1.00      0.98       110
      Consumer       1.00      0.99      0.99       100
      Criminal       0.99      0.99      0.99       102
        Family       0.98      0.96      0.97       106
        Labour       0.99      1.00      0.99        87
      Property       1.00      1.00      1.00        88
   Service Law       0.98      0.99      0.98        94
           Tax       1.00      1.00      1.00       111

      accuracy                           0.99      1000
     macro avg       0.99      0.99      0.99      1000
  weighted avg       0.99      0.99      0.99      1000



In [32]:
#find case type
import json
import numpy as np

query = "Flight delayed compensation soght"

embedding = sbert.encode([query])

prediction = classifier.predict(embedding)
probability = classifier.predict_proba(embedding)

# Get the class labels from the classifier
class_labels = classifier.classes_

# Create a dictionary to store probabilities for each class
probabilities_dict = dict(zip(class_labels, probability[0]))

# Sort probabilities in descending order
sorted_probabilities = sorted(probabilities_dict.items(), key=lambda item: item[1], reverse=True)

filtered_results = []
for case_type, prob in sorted_probabilities:
    if prob > 0.6:
        filtered_results.append({"case_type": case_type, "probability": prob})

final_probabilities_output = {}
if len(filtered_results) > 1:
    # If multiple categories > 0.6, return top 3 of them
    for item in filtered_results[:3]:
        final_probabilities_output[item['case_type']] = item['probability']
elif len(filtered_results) == 1:
    # If only one category > 0.6, return that one
    final_probabilities_output[filtered_results[0]['case_type']] = filtered_results[0]['probability']
else:
    # If no category > 0.6, return the single highest one
    top_case_type, top_prob = sorted_probabilities[0]
    final_probabilities_output[top_case_type] = top_prob

# Prepare the output as a JSON object
output_json = {
    "query": query,
    "predicted_case_type": prediction[0],
    "probabilities": final_probabilities_output
}

print(json.dumps(output_json, indent=4))

{
    "query": "Flight delayed compensation soght",
    "predicted_case_type": "Consumer",
    "probabilities": {
        "Consumer": 0.5215856527275777
    }
}


In [35]:
import joblib
import os

# Define paths for saving models
classifier_path = '/content/logistic_regression_classifier.joblib'
sbert_path = 'sbert_model_all-MiniLM-L6-v2'

# Save the Logistic Regression classifier
joblib.dump(classifier, classifier_path)
print(f"Classifier saved to {classifier_path}")

# Save the SentenceTransformer model
sbert.save(sbert_path)
print(f"SBERT model saved to {sbert_path}")

# To load them later, you would use:
# loaded_classifier = joblib.load(classifier_path)
# loaded_sbert = SentenceTransformer(sbert_path)

Classifier saved to /content/logistic_regression_classifier.joblib


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT model saved to sbert_model_all-MiniLM-L6-v2


In [37]:
import joblib
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import SGDClassifier, LogisticRegression # Import SGDClassifier
import numpy as np # Import numpy for unique classes

# --- 1. Load the existing models ---
classifier_path = '/content/logistic_regression_classifier.joblib'
sbert_path = 'sbert_model_all-MiniLM-L6-v2'

loaded_classifier = joblib.load(classifier_path)
loaded_sbert = SentenceTransformer(sbert_path)
print("Models loaded successfully.")

# --- Fix: Ensure the loaded classifier supports partial_fit for incremental training ---
# If the loaded model is LogisticRegression, it cannot perform partial_fit.
# For incremental training, we need a classifier like SGDClassifier.
# We will re-initialize `loaded_classifier` as an SGDClassifier to enable partial_fit.
# This means we are starting a new incremental training with SGDClassifier, discarding the LogisticRegression weights.
if isinstance(loaded_classifier, LogisticRegression):
    print("Warning: Loaded classifier is LogisticRegression, which does not support partial_fit.")
    print("Re-initializing as SGDClassifier for incremental training. This will discard previous LogisticRegression weights and start fresh incrementally.")

    # Get all known classes from the previously trained LogisticRegression model.
    # This is crucial for the first partial_fit call of SGDClassifier.
    all_initial_classes = loaded_classifier.classes_

    # Initialize SGDClassifier. Using 'log_loss' makes it perform logistic regression.
    # Setting random_state for reproducibility.
    loaded_classifier = SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3, random_state=42)
    # The first partial_fit call of SGDClassifier *requires* the 'classes' argument
    # to be set with all possible classes the model should ever see.
    # We will use the classes from the previously loaded LogisticRegression.

# --- 2. Load the new training data ---
# Assuming 'new_data.csv' is your new training data file
# And it has 'query' and 'case_type' columns, similar to your original data
# For demonstration, I'll create a dummy new dataframe or you can replace it with your actual new data loading.
new_data_df = pd.DataFrame({
    'query': [
        'New case about property inheritance dispute',
        'Another new case for consumer complaint about faulty product',
        'Third new case related to labor law dispute over wages'
    ],
    'case_type': ['Property', 'Consumer', 'Labour']
})
# If you have an actual CSV file for new data, use:
# new_data_df = pd.read_csv('path/to/your/new_data.csv')

print("New data loaded:")
display(new_data_df.head())

# --- 3. Prepare the new data ---
new_train_queries = new_data_df['query'].tolist()
new_train_labels = new_data_df['case_type'].tolist()

# Convert new training queries to embeddings using the loaded SBERT model
X_new_train = loaded_sbert.encode(new_train_queries)

# --- 4. Perform incremental training ---
# Now that loaded_classifier is an SGDClassifier (either loaded as such or re-initialized),
# partial_fit can be called.

# Combine initial and new classes to ensure SGDClassifier sees all possible classes
# if it's being initialized or needs to update its class knowledge.
# This step is critical for SGDClassifier's partial_fit.
if 'all_initial_classes' in locals(): # If we re-initialized from LR, we have these.
    all_combined_classes = np.unique(list(all_initial_classes) + list(new_train_labels))
else:
    # If the loaded classifier was already an SGDClassifier, its classes_ attribute should be set.
    # We combine it with new labels for robustness.
    all_combined_classes = np.unique(list(loaded_classifier.classes_) + list(new_train_labels))

loaded_classifier.partial_fit(X_new_train, new_train_labels, classes=all_combined_classes)
print("Classifier updated with new data using partial_fit.")

# --- 5. Save the updated models ---
# Save the now SGDClassifier (or updated SGDClassifier) model
joblib.dump(loaded_classifier, classifier_path)
print(f"Updated Classifier saved to {classifier_path}")

# SBERT model does not need to be re-saved if it wasn't changed
# If you loaded a pre-trained SBERT, you only need to save it once.
# If you fine-tuned SBERT (which is not done here), you would save it again.
# loaded_sbert.save(sbert_path)
# print(f"SBERT model (unchanged) saved to {sbert_path}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Models loaded successfully.
Re-initializing as SGDClassifier for incremental training. This will discard previous LogisticRegression weights and start fresh incrementally.
New data loaded:


,query,case_type
0,New case about property inheritance dispute,Property
1,Another new case for consumer complaint about ...,Consumer
2,Third new case related to labor law dispute ov...,Labour


Classifier updated with new data using partial_fit.
Updated Classifier saved to /content/logistic_regression_classifier.joblib
